## 1. Create a function to send static data to MySQL 

1.1. Create function to get cities list and its coordinates from Wikipedia

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse    # for decimal coordinates

def cities_dataframe(cities):

  city_data = []

  for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_latitude = city_soup.find(class_="latitude").get_text()
    city_longitude = city_soup.find(class_="longitude").get_text()
    country = city_soup.find(class_="infobox-data").get_text()

    # keep track of data per city
    city_data.append({"city": city,
                    "country": country,
                    "latitude": parse(city_latitude), # latitude in decimal format
                    "longitude": parse(city_longitude), # longitude in decimal format
                    })

  return pd.DataFrame(city_data)

In [2]:
# call the function
list_of_cities = ["Berlin", "Hamburg", "Munich"]

cities_df = cities_dataframe(list_of_cities)
cities_df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


In [ ]:
# Create DB
# Add query on creation cities table im Cloud SQL
DROP TABLE IF EXISTS cities;
-- Create the 'cities' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT, -- Automatically generated ID for each city
    city VARCHAR(200) NOT NULL, -- Name of the city
    country VARCHAR(100),
    latitude DECIMAL (9,6),
    longitude DECIMAL (9,6),
    PRIMARY KEY (city_id) -- Primary key to uniquely identify each city
);
SELECT * FROM cities;

1.2. Retrieving our DataFrame to Cloud SQL

In [ ]:
# External using of credentials
import os
from dotenv import load_dotenv
load_dotenv() # load .env into environment

True

In [ ]:
def connetion_string_cloud():
    schema = "cloud_gans"
    host = os.getenv("host_cloud")
    user = "root"
    password = os.getenv("password_cloud")
    port = 3306

    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    return connection_string

In [5]:
connection_string = connetion_string_cloud()

In [6]:
# insert cities_df into MySQL
cities_df.to_sql('cities',
                  con=connection_string,
                  if_exists='append',
                  index=False)

3

In [7]:
# retrieve the cities with their auto-generated IDs
pd.read_sql("""
            SELECT city_id, city
            FROM cities
            """,
            con=connection_string)

,city_id,city
0,1,Berlin
1,2,Hamburg
2,3,Munich


## 2. Create a function to send dynamic data to MySQL

2.1. Get the weather data

Add to `requirements.txt` file these modules (Cloud SQL):
* functions-framework
* SQLAlchemy
* PyMySQL
* pandas
* pysqlite3-binary
* requests

Add to `main.py` file the function and test it locally before deploying it in Cloud Run functions (Cloud SQL):

In [ ]:
# for testing and debugging the function locally or run `pip install functions-framework` in terminal
!pip install functions-framework


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2.2. Send weather data to  MySQL

In [ ]:
import functions_framework
import pandas as pd
import requests
from sqlalchemy import create_engine

@functions_framework.http
def weather_pushing(request):

    # 1. Create connection and engine
    engine = connection()

    # 2. Get cities from database
    cities_df = get_cities_data(engine)

    # 3. Get city's geolocation from databese
    cities_with_geo = get_geolocation(cities_df)

    # 4. Get the weather data for each city and store it to our database
    get_and_store_weather_data(cities_with_geo, engine)

    return 'Data successfully sent'

def connection():
    schema = "cloud_gans"
    host = os.environ.get("host_cloud")
    user = "root"
    password = os.environ.get("password_cloud")
    port = 3306
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
    return create_engine(connection_string)

def get_cities_data(engine):
    # Read the cities table into a DataFrame
    return pd.read_sql("SELECT city_id, city, country FROM cities", con=engine)

def get_geolocation(cities_df):
    API_key = os.environ.get("AeroDataBoxApi")
    geo_list = []

    for _, row in cities_df.iterrows():
        city, country, city_id = row['city'], row['country'], row['city_id']
        # geocoding URL path
        geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country}&limit=1&appid={API_key}"
        geo_response = requests.get(geo_url)

        if geo_response.status_code == 200 and geo_response.json():
            data = geo_response.json()[0]
            row['lat'] = data['lat']
            row['lon'] = data['lon']
            geo_list.append(row)

    return pd.DataFrame(geo_list)

def get_and_store_weather_data(cities_df, engine):
    API_key = os.environ.get("AeroDataBoxApi")
    weather_data = []
    for _, row in cities_df.iterrows():
            lat, lon, city_id = row['lat'], row['lon'], row['city_id']
            # weather URL path
            weather_url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_key}&units=metric"
            weather_response = requests.get(weather_url).json()

            if 'list' in weather_response:
                temp_df = pd.json_normalize(weather_response['list'])
                temp_df['city_id'] = city_id

                # unpack nested weather description safely
                if 'weather' in temp_df.columns:
                    temp_df['weather.main'] = temp_df['weather'].str[0].str.get('main')
                    temp_df['weather.description'] = temp_df['weather'].str[0].str.get('description')

                weather_data.append(temp_df)

    if weather_data:
        full_df = pd.concat(weather_data, ignore_index=True)
        column_mapping = {
            'dt_txt': 'forecast_time',
            'main.temp': 'temperature',
            'main.feels_like': 'feels_like',
            'wind.speed': 'wind_speed',
            'wind.gust': 'wind_gust',
            'visibility': 'visibility',
            'weather.main': 'weather',
            'weather.description': 'description_',
            'pop': 'precipitation_prob',
            'rain.3h': 'rain',
            'snow.3h': 'snow'
        }

    # prevent KeyError if rain/snow columns are missing from the API response
    final_df = (
        full_df.reindex(columns=list(column_mapping.keys()) + ['city_id'], fill_value=0)
        .rename(columns=column_mapping)
        .fillna(0)
    )

    final_df.to_sql(name='weathers', con=engine, if_exists='append', index=False)

2.3. To test our function, we can simulate a Flask Request object with test data. Add the following code:

In [6]:
from flask import Request
import json

# Simulate request data
request_data = {}
request = Request.from_values(data=json.dumps(request_data))

# Call the function and print the response
response = weather_pushing(request)
print(response)

Data successfully sent


2.4. Get the flights data

2.4.1. First Get the cities_airports and airports info (static data)

In [ ]:
import requests
from geopy.geocoders import Nominatim
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
def connetion_string_cloud():
    schema = "cloud_gans"
    host = os.getenv("host_cloud")
    user = "root"
    password = os.getenv("password_cloud")
    port = 3306

    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    return connection_string

In [13]:
connection_string = connetion_string_cloud()

In [14]:
API_key = os.getenv("AeroDataBoxApi")
if API_key is None:
    print("Error: API_key not found in .env file!")
else:
    print(f"Key loaded successfully: {API_key[:5]}...") # prints first 5 characters

Key loaded successfully: 68729...


In [15]:
# setup Geocoder (no API key needed for Nominatim)
geolocator = Nominatim(user_agent="my_airport_search_app")
# define cities:
cities = ['Berlin', 'Hamburg', 'Munich']
all_airports = []
for city in cities:
    # get coordinates for the city
    location = geolocator.geocode(city)
    if location:
        lat, lon = location.latitude, location.longitude
        print(f"Searching airports near {city}({lat},{lon})")
        # constract AeroBox url format
        url = f"https://aerodatabox.p.rapidapi.com/airports/search/location/{lat}/{lon}/km/50/16"
        querystring = {"withFlightInfoOnly":"true"}
        headers = {
	"X-RapidAPI-Key": API_key.strip(),
	"X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}
        response = requests.request("GET", url, headers=headers, params=querystring)
        if response.status_code == 200:
            data = response.json().get('items', [])
            all_airports.extend(data)
            print(response.json())
        else:
            print(f"Error for {city}: {response.json().get('message')}")
    else:
        print(f"Could not find coordinates for {city}")

airports_df = pd.json_normalize(all_airports)
airports_df

Searching airports near Berlin(52.5173885,13.3951309)
{'searchBy': {'lat': 52.517387, 'lon': 13.395131}, 'count': 2, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
Searching airports near Hamburg(53.550341,10.000654)
{'searchBy': {'lat': 53.550343, 'lon': 10.000654}, 'count': 1, 'items': [{'icao': 'EDDH', 'iata': 'HAM', 'name': 'Hamburg', 'shortName': 'Hamburg', 'municipalityName': 'Hamburg', 'location': {'lat': 53.6304, 'lon': 9.988229}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
Searching airports near Munich(48.1371079,11.5753822)
{'searchBy': {'lat': 48.137108, 'lon': 11.575382}, 'count'

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


In [16]:
# remove TXL from our DataFrame
clean_df = airports_df[airports_df['iata'] != 'TXL'].copy()
clean_df

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


In [17]:
# define a dictionary mapping the old names to new names
column_mapping = {
    'icao': 'airport_icao',
    'iata': 'iata',
    'name': 'airport_name',
    'shortName': 'short_name',
    'municipalityName': 'city',
    'countryCode': 'country_code',
    'timeZone': 'timezone',
    'location.lat': 'latitude',
    'location.lon': 'longitude'
       }

# apply the rename
main_df = clean_df.rename(columns=column_mapping)

# view the final result
main_df

,airport_icao,iata,airport_name,short_name,city,country_code,timezone,latitude,longitude
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100


2.4.2. Retrieving DataFrame to MySQL

In [18]:
# dataFrame for the 'airports' table
# should include: airport_icao, iata, airport_name, etc.
airports_df = main_df[['airport_icao', 'iata', 'airport_name', 'short_name',
                       'country_code', 'timezone', 'latitude', 'longitude']].drop_duplicates()

# read cities table from MySql
cities_from_sql = pd.read_sql_table('cities', con=connection_string)
# merge cities and airports to get `city_id`
merged_df = main_df.merge(cities_from_sql[['city_id', 'city']],
                              on='city',
                              how='left')

# dataFrame for the 'cities_airports' junction table
# should include: city_id, airport_icao
cities_airports_df = merged_df[['city_id', 'airport_icao']].drop_duplicates()

In [19]:
def send_airports_to_sql(airports_df, cities_airports_df, connection_string):
    # define columns for the 'airports' table
    airports_columns = [
        'airport_icao', 'iata', 'airport_name', 'short_name',
        'country_code', 'timezone', 'latitude', 'longitude'
    ]

    # define columns for the 'cities_airports' table
    cities_airports_columns = ['city_id', 'airport_icao']
    print("Current columns in airports_df:", airports_df.columns.tolist())

    try:
         # upload junction first (since DB requires this order)
        data_junction = cities_airports_df[cities_airports_columns].drop_duplicates()
        data_junction.to_sql(name='cities_airports', con=connection_string, if_exists='append', index=False)
        print(f"Success: {len(data_junction)} rows synced to 'cities_airports'.")

        # upload airports second
        data_airports = airports_df[airports_columns].drop_duplicates()
        data_airports.to_sql(name='airports', con=connection_string, if_exists='append', index=False)
        print(f"Success: {len(data_airports)} rows synced to 'airports'.")

    except Exception as e:
        print(f"SQL Error: {e}")

send_airports_to_sql(airports_df, cities_airports_df, connection_string)

Current columns in airports_df: ['airport_icao', 'iata', 'airport_name', 'short_name', 'country_code', 'timezone', 'latitude', 'longitude']
Success: 3 rows synced to 'cities_airports'.
Success: 3 rows synced to 'airports'.


2.4.3. Create a function to use it in Cloud SQL

In `requerements.txt` file:
* functions-framework==3.*
* pandas
* requests
* sqlalchemy
* pymysql

In [ ]:
import functions_framework
import requests
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import time
import re

@functions_framework.http
def flights_pushing(request):
    """
    Main Cloud Function entry point.
    """
    # 1. Setup connection and engine
    engine = connection()

    # 2. Get list of airports to check
    icao_list = get_airports_from_db(engine)

    # 3. Fetch flight data from API
    flights_df = fetch_flight_data(icao_list)

    # 4. Clean and Store
    if not flights_df.empty:
        store_flights(flights_df, engine)
        return f"Successfully processed {len(flights_df)} flights."

    return "No flight data found for the requested period."

def connection():
    schema = "cloud_gans"
    host = os.environ.get("host_cloud")
    user = "root"
    password = os.environ.get("password_cloud")
    port = 3306
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
    return create_engine(connection_string)

def get_airports_from_db(engine):
    # Fetch unique ICAOs from airports table
    airports_df = pd.read_sql("SELECT DISTINCT airport_icao FROM airports", con=engine)
    return airports_df['airport_icao'].tolist()

def fetch_flight_data(icao_list):
    API_key = os.environ.get("AeroDataBoxApi")
    headers = {
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "x-rapidapi-key": API_key
    }

    tomorrow = (datetime.now() + timedelta(days=1)).date()
    # 12-hour windows to satisfy API constraints
    time_windows = [
        (f"{tomorrow}T00:00", f"{tomorrow}T11:59"),
        (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
    ]

    all_flights = []

    for icao in icao_list:
        for start, end in time_windows:
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            querystring = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false"
            }

            response = requests.get(url, headers=headers, params=querystring)

            if response.status_code == 200:
                data = response.json()
                for item in data.get("arrivals", []):
                    all_flights.append({
                        "arrival_aeroport_icao": icao,
                        "departure_airport_icao": item.get("departure", {}).get("airport", {}).get("icao"),
                        "flight_number": item.get("number"),
                        "airline": item.get("airline", {}).get("name"),
                        "arrival_time": item.get("arrival", {}).get("scheduledTime", {}).get("local"),
                        "arrival_terminal": item.get("arrival", {}).get("terminal"),
                        "departure_city": item.get("departure", {}).get("airport", {}).get("name"),
                        "data_retrieved_on": datetime.now()
                    })
            time.sleep(1.0) # Rate limiting

    return pd.DataFrame(all_flights)

def store_flights(df, engine):
    # --- CLEANING ---
    # Remove timezone offset and handle NaNs
    df["arrival_time"] = df["arrival_time"].apply(
        lambda x: re.sub(r'[+-]\d{2}:\d{2}$', '', str(x)) if pd.notnull(x) and str(x) not in ['None', 'nan', ''] else None
    )

    # Fill missing strings to avoid SQL NULL errors
    df["arrival_terminal"] = df["arrival_terminal"].fillna("N/A")
    df["departure_city"] = df["departure_city"].fillna("Unknown")

    # Convert to proper types
    df["arrival_time"] = pd.to_datetime(df["arrival_time"], errors='coerce')
    df["data_retrieved_on"] = pd.to_datetime(df["data_retrieved_on"]).dt.floor('s')

    # Final filter for SQL schema
    sql_columns = [
        "arrival_aeroport_icao", "flight_number", "airline",
        "arrival_time", "arrival_terminal", "departure_city",
        "departure_airport_icao", "data_retrieved_on"
    ]

    # Drop rows without time and upload
    final_df = df.dropna(subset=["arrival_time"])[sql_columns]
    final_df.to_sql(name='flights', con=engine, if_exists='append', index=False)



In [23]:
# test the function locally
from flask import Request
import json

# Simulate request data
request_data = {}
request = Request.from_values(data=json.dumps(request_data))

# Call the function and print the response
response = flights_pushing(request)
print(response)

Successfully processed 873 flights.
